In [1]:
%%bash

wget -O "./protein/Apis_cerana.faa.gz" "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.1_AcerK_1.0/GCF_029169275.1_AcerK_1.0_protein.faa.gz"
gunzip "./protein/Apis_cerana.faa.gz"

--2026-11-01 10:21:08--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.1_AcerK_1.0/GCF_029169275.1_AcerK_1.0_protein.faa.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6361798 (6.1M) [application/x-gzip]
Saving to: ‘./protein/Apis_cerana.faa.gz’

     0K .......... .......... .......... .......... ..........  0% 89.5K 69s
    50K .......... .......... .......... .......... ..........  1% 34.2M 34s
   100K .......... .......... .......... .......... ..........  2%  180K 34s
   150K .......... .......... .......... .......... ..........  3%  105M 25s
   200K .......... .......... .......... .......... ..........  4%  182K 27s
   250K .......... .......... .......... .......... ..........  4% 35.1M 22s
   300K .......... .......... .......... .......... 

确定 adata 的 gene_id 与蛋白组中的 header 都是 GTF 中特定列的子集：

In [3]:
import scanpy as sc
import pandas as pd
from pyfaidx import Fasta

In [ ]:
Acer_gtf = pd.read_csv("./metadata/Apis_cerana.tsv", sep="\t") # GCF_029169275.1, not latest GCF_029169275.2!
# Acer_gtf
#! Apis_cerana.tsv: see https://github.com/zliu90693/anno_cells/tree/main/.subprojects/MetaNeighbor/metadata/Apis_cerana.tsv

/tmp/ipykernel_307261/1468964168.py:1: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  Acer_gtf = pd.read_csv("./metadata/Apis_cerana.tsv", sep="\t") # GCF_029169275.1, not latest GCF_029169275.2!


In [7]:
Acer_gtf_gid = set(Acer_gtf["gene_id"])
Acer_gtf_pid = set(Acer_gtf["protein_id"])

In [8]:
Acer_adata = sc.read_h5ad("./matrix/01_Acer_transfromHY.h5ad")

In [11]:
(Acer_adata.var.index != Acer_adata.var["gene_ids"]).sum()

0

In [12]:
Acer_adata_gid = set(Acer_adata.var["gene_ids"])

In [ ]:
Acer_protein = Fasta("./protein/Apis_cerana.faa")

In [15]:
Acer_protein_pid = set(header for header in Acer_protein.keys())

In [20]:
print(len(Acer_gtf_gid), len(Acer_adata_gid), len(Acer_gtf_gid & Acer_adata_gid))


12561 11817 11817


In [21]:
print(len(Acer_gtf_pid), len(Acer_protein_pid), len(Acer_gtf_pid & Acer_protein_pid))

29189 29188 29188


构建 protein-gene 对应关系

由于 "./protein/Apis_cerana.faa" 的格式不符合 Ensembl 的 gene_symbol: header 结构，因此不使用 SATURN 中的 map_gene_symbol_to_protein_ids.py 处理 NCBI faa，手动处理：

In [22]:
import json
import pandas as pd

Acer_gtf = pd.read_csv("metadata/Apis_cerana.tsv", sep="\t", low_memory=False)

m = Acer_gtf.dropna(subset=["gene_id", "protein_id"])[["gene_id", "protein_id"]]
m = m.drop_duplicates()
m["gene_id"] = m["gene_id"].astype(str)
m["protein_id"] = m["protein_id"].astype(str)

gene_to_proteins = {}
for gene, sub in m.groupby("gene_id"):
    gene_to_proteins[gene] = sorted(set(sub["protein_id"]))

with open("./gene_protein_map/Acer.gene_id_to_protein_ID.json", "w") as f:
    json.dump(gene_to_proteins, f, indent=4, sort_keys=True)

生成 SATURN-ready 的 h5ad

In [1]:
import scanpy as sc
import numpy as np

/home/liuzy/Software/miniconda3/envs/saturn/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [2]:
Acer = sc.read_h5ad("./matrix/01_Acer_transfromHY.h5ad")
Acer.X = Acer.layers["counts"].copy().astype(np.float32) # 输入的矩阵必须是原始计数
for k in list(Acer.layers.keys()): # 删除不需要的 layers 减少空间
    del Acer.layers[k]

In [6]:
Acer.obs["saturn_label"] = "cluster" + Acer.obs["leiden_res1.00"].astype(str)
Acer.obs["saturn_batch"] = Acer.obs["batch"].astype(str)

In [7]:
Acer.obs

,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,batch,leiden_res0.25,leiden_res0.50,leiden_res1.00,HY_barcode,celltype_transfrom_HY,saturn_label,saturn_batch
GTAACCACATCATCCC-1_queen_rep1_singlet,7048,8.860641,110980.0,11.617114,14.631465,1911.0,7.555905,1.721932,queen_rep1_singlet,1,1,2,GTAACCACATCATCCC-1_queen_rep1,unknown,cluster2,queen_rep1_singlet
GGCAGTCCACCAATTG-1_queen_rep1_singlet,6956,8.847504,94809.0,11.459630,19.844108,2668.0,7.889459,2.814079,queen_rep1_singlet,1,7,7,GGCAGTCCACCAATTG-1_queen_rep1,unknown,cluster7,queen_rep1_singlet
GACCTTCTCACTGTTT-1_queen_rep1_singlet,6953,8.847072,91343.0,11.422388,13.937576,2423.0,7.793174,2.652639,queen_rep1_singlet,1,7,7,GACCTTCTCACTGTTT-1_queen_rep1,unknown,cluster7,queen_rep1_singlet
TGAGGTTCACGTACTA-1_queen_rep1_singlet,6749,8.817298,90117.0,11.408875,12.586970,1162.0,7.058758,1.289435,queen_rep1_singlet,1,7,7,TGAGGTTCACGTACTA-1_queen_rep1,unknown,cluster7,queen_rep1_singlet
TCTCCGAAGACCACGA-1_queen_rep1_singlet,6718,8.812695,87982.0,11.384899,12.691232,1936.0,7.568896,2.200450,queen_rep1_singlet,1,7,7,TCTCCGAAGACCACGA-1_queen_rep1,unknown,cluster7,queen_rep1_singlet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AGATGAACAGCTTCCT-1_drone_rep2_singlet,11,2.484907,210.0,5.351858,100.000000,206.0,5.332719,98.095238,drone_rep2_singlet,4,29,35,AGATGAACAGCTTCCT-1_drone_rep2,unknown,cluster35,drone_rep2_singlet
TGATTCTCAGCTATTG-1_drone_rep2_singlet,12,2.564949,220.0,5.398163,100.000000,220.0,5.398163,100.000000,drone_rep2_singlet,4,29,35,TGATTCTCAGCTATTG-1_drone_rep2,unknown,cluster35,drone_rep2_singlet
AAGGAATAGGTACATA-1_drone_rep2_singlet,12,2.564949,215.0,5.375278,100.000000,214.0,5.370638,99.534884,drone_rep2_singlet,4,29,35,AAGGAATAGGTACATA-1_drone_rep2,unknown,cluster35,drone_rep2_singlet
CATACTTGTCCTACGG-1_drone_rep2_singlet,11,2.484907,227.0,5.429346,100.000000,227.0,5.429346,100.000000,drone_rep2_singlet,4,29,35,CATACTTGTCCTACGG-1_drone_rep2,unknown,cluster35,drone_rep2_singlet


In [8]:
assert not Acer.obs["saturn_label"].isna().any()
Acer.write_h5ad("./matrix/Acer_saturn_input.h5ad")